<a href="https://colab.research.google.com/github/ReverieRiver/deepseek-qwen3-finetune/blob/main/nb/DeepSeek_R1_0528_Qwen3_(8B)_GRPO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

To run this, press "*Runtime*" and press "*Run all*" on a **free** Tesla T4 Google Colab instance!
<div class="align-center">
<a href="https://unsloth.ai/"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
<a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord button.png" width="145"></a>
<a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a> Join Discord if you need help + ⭐ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐
</div>

To install Unsloth on your local device, follow [our guide](https://unsloth.ai/docs/get-started/install). This notebook is licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme).

You will learn how to do [data prep](#Data), how to [train](#Train), how to [run the model](#Inference), & how to save it

### News

Introducing **[Unsloth Desktop](https://unsloth.ai/docs/desktop)**, the first desktop app to run and train models. Free and open-source for macOS, Windows and Linux. [GitHub](https://github.com/unslothai/unsloth) • [Download](https://unsloth.ai/download)

<p>
<a href="https://unsloth.ai/docs/desktop"><img src="https://raw.githubusercontent.com/unslothai/notebooks/refs/heads/main/assets/unsloth-qwen3-8.png" width="350" alt="Introducing Unsloth Desktop"></a>
</p>

Train MoEs - DeepSeek, GLM, Qwen and gpt-oss 12x faster with 35% less VRAM. [Blog](https://unsloth.ai/docs/new/faster-moe)

Ultra Long-Context Reinforcement Learning is here with 7x more context windows! [Blog](https://unsloth.ai/docs/new/grpo-long-context)

New in Reinforcement Learning: [FP8 RL](https://unsloth.ai/docs/new/fp8-reinforcement-learning) • [Vision RL](https://unsloth.ai/docs/new/vision-reinforcement-learning-vlm-rl) • [Standby](https://unsloth.ai/docs/basics/memory-efficient-rl) • [gpt-oss RL](https://unsloth.ai/docs/new/gpt-oss-reinforcement-learning)

Visit our docs for all our [model uploads](https://unsloth.ai/docs/get-started/unsloth-model-catalog) and [notebooks](https://unsloth.ai/docs/get-started/unsloth-notebooks).

### Installation

In [1]:
%%capture
import os
os.environ["UNSLOTH_VLLM_STANDBY"] = "1" # [NEW] Extra 30% context lengths!
if "COLAB_" not in "".join(os.environ.keys()):
    # If you're not in Colab, just use pip install or uv pip install
    !pip install unsloth vllm
else:
    pass # For Colab / Kaggle, we need extra instructions hidden below \/

In [2]:
#@title Colab Extra Install { display-mode: "form" }
%%capture
import os
!pip install --upgrade -qqq uv
if "COLAB_" not in "".join(os.environ.keys()):
    # If you're not in Colab, just use pip install!
    !pip install unsloth vllm
else:
    try: import numpy, PIL; _numpy = f'numpy=={numpy.__version__}'; _pil = f'pillow=={PIL.__version__}'
    except: _numpy = "numpy"; _pil = "pillow"
    try: import subprocess; is_t4 = "Tesla T4" in str(subprocess.check_output(["nvidia-smi"]))
    except: is_t4 = False
    _vllm, _triton = ('vllm==0.11.2', 'triton') if is_t4 else ('vllm==0.15.1', 'triton')
    !uv pip install -qqq --upgrade {_vllm} {_numpy} {_pil} torchvision bitsandbytes xformers unsloth
    !uv pip install -qqq {_triton}
    try:
        import importlib.metadata as _md; _torch_v = tuple(int(_p) for _p in _md.version("torch").split("+")[0].split(".")[:2])
    except Exception:
        _torch_v = ()
    # torchao 0.18.0 imports torch.nn.functional.ScalingType, added in torch 2.10; peft >= 0.19 needs the 0.16.0 floor.
    _torchao = "torchao>=0.16.0" if _torch_v >= (2, 10) else "torchao>=0.16.0,<0.18.0"
    !uv pip install -qqq --no-deps --upgrade "{_torchao}"
!uv pip install transformers==4.56.2
!uv pip install --no-deps trl==0.22.2

### Unsloth

Goal: To convert `DeepSeek-R1-0528-Qwen3-8B` into a reasoning model specialized for roleplaying using the `ZenMoore/RoleBench` dataset.

In [3]:
!pip install langid -qq

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 18.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [4]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048 # Can increase for longer reasoning traces
lora_rank = 32 # Larger rank = smarter, but slower

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/DeepSeek-R1-0528-Qwen3-8B",
    max_seq_length = max_seq_length,
    load_in_4bit = True, # False for LoRA 16bit
    fast_inference = True, # Enable vllm fast inference
    max_lora_rank = lora_rank,
    gpu_memory_utilization = 0.9, # Reduce if out of memory
)

model = FastLanguageModel.get_peft_model(
    model,
    r = lora_rank, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha = lora_rank*2, # *2 speeds up training
    use_gradient_checkpointing = "unsloth", # Reduces memory usage
    random_state = 3407,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/usr/local/lib/python3.13/dist-packages/unsloth/_gpu_init.py:357: UserWarning: torchcodec 0.11.0+cu128 is incompatible with torch 2.9.0+cu128; install a matching build with `pip install --index-url https://download.pytorch.org/whl/cu128 'torchcodec>=0.9,<0.10.0'`.
  disable_torchcodec_if_broken()
/usr/local/lib/python3.13/dist-packages/unsloth/_gpu_init.py:357: UserWarning: Unsloth: torchcodec is installed but cannot load its native libraries although FFmpeg is on the loader path; likely an FFmpeg major it does not support (it takes 4 to 8), a missing CUDA NPP runtime (nvidia-npp), or a build that does not match this torch; audio datasets decode through soundfile and PyAV instead (wav/flac/mp3/ogg, m4a/aac/webm).
  disable_torchcodec_if_broken()


ERROR 09-24 22:40:41 [fa_utils.py:64] Cannot use FA version 2 is not supported due to FA2 is only supported on devices with compute capability >= 8
🦥 Unsloth Zoo will now patch everything to make training faster!


Unrecognized keys in `rope_scaling` for 'rope_type'='yarn': {'attn_factor'}


INFO 09-24 22:41:08 [vllm_utils.py:808] Unsloth: Patching vLLM v1 graph capture
==((====))==  Unsloth 2026.9.11: Fast Qwen3 patching. Transformers: 4.56.2. vLLM: 0.11.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


...

`torch_dtype` is deprecated! Use `dtype` instead!
Unrecognized keys in `rope_scaling` for 'rope_type'='yarn': {'attn_factor'}


Unsloth: Just some info: will skip parsing ['post_layernorm', 'norm2', 'layer_norm2', 'attention_norm', 'ffn_norm', 'post_per_layer_input_norm', 'q_norm', 'pre_feedforward_layernorm', 'layer_norm1', 'post_feedforward_layernorm', 'input_layernorm', 'post_attention_layernorm', 'k_norm', 'norm1', 'norm']


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Performing substitution for additional_keys=set()
Unsloth: Just some info: will skip parsing ['post_layernorm', 'norm2', 'layer_norm2', 'attention_norm', 'ffn_norm', 'post_per_layer_input_norm', 'q_norm', 'pre_feedforward_layernorm', 'layer_norm1', 'cross_attn_input_layernorm', 'cross_attn_post_attention_layernorm', 'post_feedforward_layernorm', 'input_layernorm', 'post_attention_layernorm', 'k_norm', 'norm1', 'norm']


Unsloth 2026.9.11 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


### GRPO Chat Template

Distill Qwen3 from Deepseek has a chat template that is used to format the input and output of the model. This is used to make the model output in a chat format. Including the reasoning step. We have to use that chat template since the model is trained using it.

Let's see how our chat template behaves on an example:

In [6]:
reasoning_start = None
reasoning_end = None
user_token = None
assistant_token = None

for token in tokenizer.get_added_vocab().keys():
    if "think" in token and "/" in token:
        reasoning_end = token
    elif "think" in token:
        reasoning_start = token
    elif "user" in token:
        user_token = token
    elif "assistant" in token:
        assistant_token = token

system_prompt = \
f"""You are a helpful roleplaying assistant.
Think step by step and provide your roleplay response."""
system_prompt

'You are a helpful roleplaying assistant.\nThink step by step and provide your roleplay response.'

In [7]:
print(tokenizer.apply_chat_template([
    {"role" : "user", "content" : "You are a wise old wizard. What spell would you cast to cheer up a sad dragon?"},
    {"role" : "assistant", "content" : f"<think>A sad dragon? Such a rare and delicate problem! I must consider a spell that brings joy without startling such a magnificent beast.</think>I would conjure a 'Whispering Lullaby of Laughter', a gentle enchantment that fills the air with the faint, melodious echoes of happy memories and soft giggles, slowly lifting the dragon's spirits and coaxing forth a warm, rumbling chuckle."},
], tokenize = False, add_generation_prompt = True))

<｜begin▁of▁sentence｜><｜User｜>You are a wise old wizard. What spell would you cast to cheer up a sad dragon?<｜Assistant｜>I would conjure a 'Whispering Lullaby of Laughter', a gentle enchantment that fills the air with the faint, melodious echoes of happy memories and soft giggles, slowly lifting the dragon's spirits and coaxing forth a warm, rumbling chuckle.<｜end▁of▁sentence｜><｜Assistant｜>


### Data Prep
<a name="Data"></a>

We're using Hugging Face's [Open R1 Math dataset](https://huggingface.co/datasets/open-r1/DAPO-Math-17k-Processed). You can also utilize OpenAI's famous [GSM8K dataset](https://huggingface.co/datasets/openai/gsm8k)

In [14]:
from datasets import load_dataset
dataset = load_dataset("ZenMoore/RoleBench", data_files={"train": "rolebench-eng/instruction-generalization/general/train.jsonl"}, split = "train")
dataset

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['role', 'question', 'generated'],
    num_rows: 112142
})

Let's look at the first row of the `RoleBench` dataset:

In [15]:
dataset[0]["question"]

'Compose a 1-2 sentence slogan for a brand that specializes in outdoor lifestyle apparel.\n'

In [16]:
dataset[0]["generated"]

['"Conquer the elements with Quantum Wear: where Mother Nature meets Motherboard."',
 '"Experience the elements, optimized. Wear the genius of comfort and durability, because the world is your laboratory."',
 '"Because adapting to the outside world shouldn\'t only be a theoretical concept: \'Outward Boundaries – crossing the line between indoor comfort and outdoor freedom.\' Bazinga!"',
 '"Discover a world beyond four walls. Our apparel is tested under conditions harsher than most of your mother\'s meatloaves."',
 '"Whether you\'re scaling the highest peak or simply strolling in the park, our apparel affirms that science is universal but so is comfort."']

For the RoleBench dataset, the 'output' field directly contains the desired response, so we can use it as is.

In [17]:
def extract_hash_answer(text):
    # For RoleBench, the 'generated' is directly the answer.
    return text
extract_hash_answer(dataset[0]["generated"])

['"Conquer the elements with Quantum Wear: where Mother Nature meets Motherboard."',
 '"Experience the elements, optimized. Wear the genius of comfort and durability, because the world is your laboratory."',
 '"Because adapting to the outside world shouldn\'t only be a theoretical concept: \'Outward Boundaries – crossing the line between indoor comfort and outdoor freedom.\' Bazinga!"',
 '"Discover a world beyond four walls. Our apparel is tested under conditions harsher than most of your mother\'s meatloaves."',
 '"Whether you\'re scaling the highest peak or simply strolling in the park, our apparel affirms that science is universal but so is comfort."']

Let's map the dataset! and see the first row:

In [18]:
dataset = dataset.map(lambda x: {
    "prompt" : [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": x["question"]},
    ],
    "answer": x["generated"],
})
dataset[0]

Map:   0%|          | 0/112142 [00:00<?, ? examples/s]

{'role': 'Sheldon Cooper',
 'question': 'Compose a 1-2 sentence slogan for a brand that specializes in outdoor lifestyle apparel.\n',
 'generated': ['"Conquer the elements with Quantum Wear: where Mother Nature meets Motherboard."',
  '"Experience the elements, optimized. Wear the genius of comfort and durability, because the world is your laboratory."',
  '"Because adapting to the outside world shouldn\'t only be a theoretical concept: \'Outward Boundaries – crossing the line between indoor comfort and outdoor freedom.\' Bazinga!"',
  '"Discover a world beyond four walls. Our apparel is tested under conditions harsher than most of your mother\'s meatloaves."',
  '"Whether you\'re scaling the highest peak or simply strolling in the park, our apparel affirms that science is universal but so is comfort."'],
 'prompt': [{'role': 'system',
   'content': 'You are a helpful roleplaying assistant.\nThink step by step and provide your roleplay response.'},
  {'role': 'user',
   'content': 'Com

We create a regex format to match the reasoning sections and answers:

In [20]:
import re

# Add optional EOS token matching
solution_end_regex = rf"{reasoning_end}(.*)"

match_format = re.compile(solution_end_regex, re.DOTALL)
# Removed implicit display of match_format to avoid AttributeError
# print(match_format.pattern) # Uncomment to see the compiled pattern

We verify it works:

In [21]:
match_format.findall(
    "Let me think!</think>"\
    f"Hence, the solution is 2.",
)

['Hence, the solution is 2.']

In [22]:
match_format.findall(
    "<think>Let me think!</think>"\
    f"\n\nHence, the solution is 2",
)

['\n\nHence, the solution is 2']

We now want to create a reward function to match the format exactly - we reward it with 3 points if it succeeds:

In [23]:
def match_format_exactly(completions, **kwargs):
    scores = []
    for completion in completions:
        score = 0
        response = completion[0]["content"]
        # Match if format is seen exactly!
        if match_format.search(response) is not None: score += 3.0
        scores.append(score)
    return scores

If it fails, we want to reward the model if it at least follows the format partially, by counting each symbol:

In [24]:
def match_format_approximately(completions, **kwargs):
    scores = []
    for completion in completions:
        score = 0
        response = completion[0]["content"]
        # Count how many keywords are seen - we penalize if too many!
        # If we see 1, then plus some points!

        # No need to reward the think tag since we always prepend it!
        score += 0.5 if response.count(reasoning_start) == 1 else -1.0
        score += 0.5 if response.count(reasoning_end)   == 1 else -1.0
        scores.append(score)
    return scores

def check_answer(prompts, completions, answer, **kwargs):
    # This function was for checking numerical answers, not applicable for roleplaying.
    # Returning a neutral score for now.
    scores = [0.0] * len(completions)
    return scores

In [25]:
def check_answer(prompts, completions, answer, **kwargs):
    # This function was for checking numerical answers, not applicable for roleplaying.
    # Returning a neutral score for now.
    scores = [0.0] * len(completions)
    return scores

### Language Consistency Reward (Removed)
This section was for enforcing Bahasa Indonesia and is no longer needed.

In [26]:
import langid

def get_lang(text: str) -> str:
    if not text:
        return "und"
    lang, _ = langid.classify(text)
    return lang

In [27]:
def format_and_language_reward_func(completions, **kwargs):
    scores = []

    for completion_item in completions:
        if not completion_item or not isinstance(completion_item[0], dict) or "content" not in completion_item[0]:
            scores.append(-5.0)
            print(f"Warning: Malformed completion item, assigning default low score: {completion_item}")
            continue

        content = completion_item[0]["content"]

        lang = get_lang(content)

        if lang == 'en':
            score = 5.0
        else:
            # Penalize any non-English language
            score = -5.0

        scores.append(score)

    return scores

In [ ]:
# Example usage for language reward (removed)

[-3.0, -3.0]

The `check_numbers` function is for comparing numerical answers, which is not applicable for roleplaying. We will keep the `match_format_exactly` and `match_format_approximately` for checking adherence to the thinking format.

In [ ]:
global PRINTED_TIMES
PRINTED_TIMES = 0
global PRINT_EVERY_STEPS
PRINT_EVERY_STEPS = 5

def check_numbers(prompts, completions, answer, **kwargs):
    # This function was for checking numerical answers, not applicable for roleplaying.
    # Returning a neutral score for now.
    scores = [0.0] * len(completions)
    # Optionally, you can still print debug info if needed, but numerical comparison is disabled.
    global PRINTED_TIMES
    global PRINT_EVERY_STEPS
    question = prompts[0][-1]["content"]
    responses = [completion[0]["content"] for completion in completions]

    if PRINTED_TIMES % PRINT_EVERY_STEPS == 0:
        print(
            '*'*20 + f"Question:\n{question}", f"\nAnswer:\n{answer[0]}", f"\nResponse:\n{responses[0]}"
        )
    PRINTED_TIMES += 1
    return scores

Get the top 90% prompt length so we don't accidentally truncate them!

Ie we'll remove the top 10% long prompts.

In [28]:
tokenized = dataset.map(
    lambda x: {"tokens" : tokenizer.apply_chat_template(x["prompt"], add_generation_prompt = True, tokenize = True)},
    batched = True,
)
print(tokenizer.decode(tokenized[0]["tokens"]))
tokenized = tokenized.map(lambda x: {"L" : len(x["tokens"])})

import numpy as np
maximum_length = int(np.quantile(tokenized["L"], 0.9))
print("Max Length = ", maximum_length)

# Filter only samples smaller than 90% max length
dataset = dataset.select(np.where(np.array(tokenized["L"]) <= maximum_length)[0])
del tokenized

Map:   0%|          | 0/112142 [00:00<?, ? examples/s]

<｜begin▁of▁sentence｜>You are a helpful roleplaying assistant.
Think step by step and provide your roleplay response.<｜User｜>Compose a 1-2 sentence slogan for a brand that specializes in outdoor lifestyle apparel.
<｜Assistant｜>


Map:   0%|          | 0/112142 [00:00<?, ? examples/s]

Max Length =  52


<a name="Train"></a>
### Train the model

Now set up GRPO Trainer and all configurations!

In [29]:
max_prompt_length = maximum_length + 1 # + 1 just in case!
max_completion_length = max_seq_length - max_prompt_length

from vllm import SamplingParams
vllm_sampling_params = SamplingParams(
    min_p = 0.01,
    top_p = 0.95,
    top_k = -1,
    seed = -1,
    stop = [tokenizer.eos_token],
    include_stop_str_in_output = True,
)

from trl import GRPOConfig, GRPOTrainer
training_args = GRPOConfig(
    vllm_sampling_params = vllm_sampling_params,
    temperature = 1.0,
    learning_rate = 5e-6,
    weight_decay = 0.001,
    warmup_ratio = 0.1,
    lr_scheduler_type = "linear",
    optim = "adamw_8bit",
    logging_steps = 1,
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 1, # Increase to 4 for smoother training
    num_generations = 4, # Decrease if out of memory
    max_prompt_length = max_prompt_length,
    max_completion_length = max_completion_length,
    num_train_epochs = 1, # Set to 1 for a full training run
    save_strategy = "epoch",
    save_total_limit = 1,
    # max_steps = 100,
    # save_steps = 100,
    report_to = "none", # Can use Weights & Biases
    output_dir = "outputs",

    # For optional training + evaluation
    # fp16_full_eval = True,
    # per_device_eval_batch_size = 4,
    # eval_accumulation_steps = 1,
    # eval_strategy = "steps",
    # eval_steps = 1,
)

Unsloth: We now expect `per_device_train_batch_size` * `gradient_accumulation_steps` * `world_size` to be a multiple of `num_generations`.
We will change the batch size of 1 to the `num_generations` of 4


And let's run the trainer! If you scroll up, you'll see a table of rewards. The goal is to see the `reward` column increase!

You might have to wait 150 to 200 steps for any action. You'll probably get 0 reward for the first 100 steps. Please be patient!

| Step | Training Loss | reward    | reward_std | completion_length | kl       |
|------|---------------|-----------|------------|-------------------|----------|
| 1    | 0.000000      | 0.125000  | 0.000000   | 200.000000        | 0.000000 |
| 2    | 0.000000      | 0.072375  | 0.248112   | 200.000000        | 0.000000 |
| 3    | 0.000000      | -0.079000 | 0.163776   | 182.500000        | 0.000005 |

In [ ]:
# For optional training + evaluation
# new_dataset = dataset.train_test_split(test_size = 0.01)

trainer = GRPOTrainer(
    model = model,
    processing_class = tokenizer,
    reward_funcs = [
        match_format_exactly,
        match_format_approximately,
        format_and_language_reward_func, # Re-added with English enforcement
        # check_answer, # Removed as human evaluation will be done offline
        # check_numbers, # Removed as it's math-specific
    ],
    args = training_args,
    train_dataset = dataset,

    # For optional training + evaluation
    # train_dataset = new_dataset["train"],
    # eval_dataset = new_dataset["test"],
)
trainer.train() # Uncommented to run the training

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 101,154 | Num Epochs = 1 | Total steps = 101,154
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 1 x 1) = 4
 "-____-"     Trainable parameters = 87,293,952 of 8,278,029,312 (1.05% trained)


WARNING 09-24 23:40:12 [processor.py:246] vLLM has deprecated support for supporting different tokenizers for different LoRAs. By default, vLLM uses base model's tokenizer. If you are using a LoRA with its own tokenizer, consider specifying `--tokenizer [lora_path]` to use the LoRA tokenizer.
Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,kl,rewards / match_format_exactly / mean,rewards / match_format_exactly / std,rewards / match_format_approximately / mean,rewards / match_format_approximately / std,rewards / format_and_language_reward_func / mean,rewards / format_and_language_reward_func / std
1,0.000000,4.500000,0.000000,1995.000000,1995.000000,1995.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-0.500000,0.000000,5.000000,0.000000
2,0.306900,7.875000,2.250000,1236.250000,616.000000,1995.000000,0.250000,983.333374,616.000000,1292.000000,0.000022,2.250000,1.500000,0.625000,0.750000,5.000000,0.000000
3,0.000000,9.000000,0.000000,570.750000,287.000000,871.000000,0.000000,570.750000,287.000000,871.000000,0.000027,3.000000,0.000000,1.000000,0.000000,5.000000,0.000000
4,0.536100,7.875000,2.250000,962.750000,313.000000,1995.000000,0.250000,618.666687,313.000000,933.000000,0.000019,2.250000,1.500000,0.625000,0.750000,5.000000,0.000000
5,0.490500,7.875000,2.250000,1007.000000,375.000000,1995.000000,0.250000,677.666687,375.000000,874.000000,0.000022,2.250000,1.500000,0.625000,0.750000,5.000000,0.000000
6,0.000000,9.000000,0.000000,926.000000,405.000000,1715.000000,0.000000,926.000000,405.000000,1715.000000,0.000020,3.000000,0.000000,1.000000,0.000000,5.000000,0.000000
7,0.000000,9.000000,0.000000,586.250000,213.000000,1067.000000,0.000000,586.250000,213.000000,1067.000000,0.000001,3.000000,0.000000,1.000000,0.000000,5.000000,0.000000
8,0.532100,7.875000,2.250000,966.500000,418.000000,1995.000000,0.250000,623.666687,418.000000,824.000000,0.000027,2.250000,1.500000,0.625000,0.750000,5.000000,0.000000
9,0.000000,4.500000,0.000000,1995.000000,1995.000000,1995.000000,1.000000,0.000000,0.000000,0.000000,0.000056,0.000000,0.000000,-0.500000,0.000000,5.000000,0.000000
10,0.000000,9.000000,0.000000,820.500000,224.000000,1595.000000,0.000000,820.500000,224.000000,1595.000000,0.000018,3.000000,0.000000,1.000000,0.000000,5.000000,0.000000


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


<a name="Inference"></a>
### Inference
Now let's try the model we just trained! First, let's first try the model without any GRPO trained:

In [ ]:
text = "You are a knight protecting a princess. What is your response when a dragon attacks?"

from vllm import SamplingParams
sampling_params = SamplingParams(
    temperature = 1.0,
    top_k = 50,
    max_tokens = 1024,
)
output = model.fast_generate(
    [text],
    sampling_params = sampling_params,
    lora_request = None,
)[0].outputs[0].text

output

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

' | Socratic\nWhat is the sqrt of 101?\nAlgebra\nQuestion\nVincent A.\nAnswer\n10.049875, or as an irrational number, it is not integer, I see what you mean though.\nExplanation:\nsqrt(100) = 10, sqrt(121)=11, so sqrt(101) is between 10 and 11, and there is not integer between 10 and 11, so it is irrational.\nWe can leave it as sqrt(101) or give the approximate value.\nThe question asks "what is the sqrt of 101", and it is a math problem, so probably they want to know if it is integer or not.\nOr perhaps do some calculations.\nI think the intended answer is that it is irrational and not an integer, but since 10^2=100 and 11^2=121, and 10^2 <101<11^2, so no integer square.\nI could also use calculator, but I think since it\'s a math problem, perhaps we need to show'

And now with the LoRA we just trained with GRPO - we first save the LoRA first!

In [ ]:
model.save_lora("grpo_lora")

Unrecognized keys in `rope_scaling` for 'rope_type'='yarn': {'attn_factor'}

Verify LoRA is actually trained!

In [ ]:
from safetensors import safe_open

tensors = {}
with safe_open("grpo_lora/adapter_model.safetensors", framework = "pt") as f:
    # Verify both A and B are non zero
    for key in f.keys():
        tensor = f.get_tensor(key)
        n_zeros = (tensor == 0).sum() / tensor.numel()
        assert(n_zeros.item() != tensor.numel())

Now we load the LoRA and test. We tested without using our custom system prompt which should not (or minimal) affect toward the model's original reasoning ability.:

In [ ]:
messages = [
    {"role": "user",   "content": "You are a medieval merchant selling exotic spices. A customer asks for your rarest spice. How do you respond?"},
]

text = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True, # Must add for generation
    tokenize = False,
)
from vllm import SamplingParams
sampling_params = SamplingParams(
    temperature = 1.0,
    top_k = 50,
    max_tokens = 2048,
)
output = model.fast_generate(
    text,
    sampling_params = sampling_params,
    lora_request = model.load_lora("grpo_lora"),
)[0].outputs[0].text

output

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

"<think>\nI have this equation: (x + 2)² = 0. It looks simple, but I need to solve for x. Since it's a squared term equal to zero, that means the thing inside the parentheses must be zero because only zero squared is zero.\n\nSo, if (x + 2)² = 0, then x + 2 must be equal to zero. Because if x + 2 were anything else, say 1, squared is 1, which is not zero. Or -1, squared is also 1, not zero. So only when x + 2 is zero, the square is zero.\n\nSo, x + 2 = 0, which means x = -2.\n\nI think that's it. Let me verify by plugging it back into the equation.\n\nx = -2, so ( -2 + 2 )² = (0)² = 0, which equals 0. Perfect.\n\nI recall that in algebra, this is related to the zero product property or something. Basically, if a product is zero, then one of the factors must be zero. Here, it's not a product, but"

Next, let's test using our system prompt which should use the new language :

In [ ]:
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user",   "content": "You are a wizard in a fantasy realm. Describe a new spell you just invented."},
]

text = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True, # Must add for generation
    tokenize = False,
)
from vllm import SamplingParams
sampling_params = SamplingParams(
    temperature = 1.0,
    top_k = 50,
    max_tokens = 2048,
)
output = model.fast_generate(
    text,
    sampling_params = sampling_params,
    lora_request = model.load_lora("grpo_lora"),
)[0].outputs[0].text

output

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

"<think>\nI need to solve the equation (x + 2)^2 = 0. It's a quadratic equation, but it's squared, so it might be simpler. Let me think about this in Bahasa Indonesia first.\n\nPermulaan: Saya diberikan persamaan (x + 2)^2 = 0, dan saya harus menyelesaikannya. Saya perlu mencari nilai x yang memenuhi persamaan ini.\n\nSaya mulai dengan merasakan apa itu persamaan. Ini adalah persamaan kuadrat karena ada eksponen 2, tapi tidak semua persamaan kuadrat memiliki dua solusi; beberapa mungkin memiliki solusi ganda.\n\nSaya tahu bahwa jika sesuatu dikalikan dengan dirinya sendiri dan hasilnya nol, maka satu di antaranya harus nol. Jadi, untuk (x + 2)^2 = 0, itu berarti x + 2 harus sama dengan 0, karena jika x + 2 tidak"

### Comparison of language usage (Removed)
This section was for comparing language usage which is no longer relevant as the Indonesian language enforcement has been removed.

In [ ]:
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user",   "content": "You are a space explorer who just landed on an alien planet. Describe what you see."},
]

text = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True, # Must add for generation
    tokenize = False,
)
from vllm import SamplingParams
sampling_params = SamplingParams(
    temperature = 1.0,
    top_k = 50,
    max_tokens = 2048,
)
output = model.fast_generate(
    text,
    sampling_params = sampling_params,
    lora_request = None,
)[0].outputs[0].text

output

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

'<think>\nBaik, mari kita selesaikan persamaan kuadrat sederhana ini. Soalnya adalah: Solve (x + 2)^2 = 0.\n\nPertama, saya perlu memahami apa yang diminta. Ini adalah persamaan kuadrat yang diatur dalam bentuk kuadrat. Saya harus mencari nilai dari x yang memenuhi persamaan tersebut.\n\nSaya tahu bahwa ketika suatu bilangan kuadrat sama dengan nol, itu berarti bilangan tersebut adalah nol. Jadi, dalam hal ini, (x + 2)^2 = 0 berarti kuadrat dari (x + 2) adalah nol. Maka, untuk kuadrat suatu bilangan sama dengan nol, bilangan tersebut haruslah nol itu sendiri.\n\nOleh karena itu, (x + 2) haruslah sama dengan nol. Jadi, x + 2 = 0.\n\nSekarang, untuk mencari x, saya'

Let's take 20 samples, and compare the the amount of using our LoRA and not using it, and see which one has better amount of correct language

In [ ]:
sample_dataset = dataset.shuffle(seed = 3407).select(range(20))
sample_dataset

Dataset({
    features: ['prompt', 'solution', 'data_source', 'source_prompt', 'ability', 'reward_model', 'extra_info', 'answer'],
    num_rows: 20
})

### Human Evaluation of Roleplay Responses

This section allows for qualitative human assessment of the model's roleplay responses. You will be presented with a prompt and the model's generated response, and you can assign a score and provide comments.


In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML

# Select a small number of samples for human evaluation
human_eval_dataset = dataset.shuffle(seed=42).select(range(5))

print(f"Preparing {len(human_eval_dataset)} samples for human evaluation.")

In [ ]:
# Store widgets and collected scores
scoring_widgets = []
human_scores_data = []

# Define sampling parameters for generation
# Using the same parameters as the inference section
from vllm import SamplingParams

sampling_params_eval = SamplingParams(
    temperature = 1.0,
    top_k = 50,
    max_tokens = 1024,
)

for i, sample in enumerate(human_eval_dataset):
    print(f"\n--- Sample {i+1}/{len(human_eval_dataset)} ---")

    # Original Prompt
    messages = sample["prompt"]
    display(HTML(f"<b>Prompt:</b>"))
    for msg in messages:
        display(HTML(f"<div style='margin-left: 20px;'><b>{msg['role'].capitalize()}:</b> {msg['content']}</div>"))

    # Generate model response with LoRA
    text_input = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt = True,
        tokenize = False,
    )

    model_response = model.fast_generate(
        text_input,
        sampling_params = sampling_params_eval,
        lora_request = model.load_lora("grpo_lora"),
    )[0].outputs[0].text

    display(HTML(f"<b>Model Response:</b> <div style='margin-left: 20px; background-color: #f0f0f0; padding: 10px; border-radius: 5px;'>{model_response}</div>"))

    # Create scoring widgets
    score_slider = widgets.IntSlider(
        value=3,
        min=1,
        max=5,
        step=1,
        description='Score (1-5):',
        disabled=False,
        continuous_update=False,
        orientation='horizontal',
        readout=True,
        readout_format='d',
    )
    comment_textarea = widgets.Textarea(
        value='',
        placeholder='Add any comments here (e.g., character consistency, creativity, relevance)',
        description='Comments:',
        disabled=False
    )

    display(score_slider, comment_textarea)
    scoring_widgets.append({
        'sample_id': i,
        'prompt': messages,
        'model_response': model_response,
        'score_widget': score_slider,
        'comment_widget': comment_textarea
    })

collect_button = widgets.Button(description="Collect All Scores")
output_area = widgets.Output()

def on_button_click(b):
    global human_scores_data
    human_scores_data = [] # Clear previous scores if any
    with output_area:
        output_area.clear_output()
        print("Collecting scores...")
        for item in scoring_widgets:
            human_scores_data.append({
                'sample_id': item['sample_id'],
                'prompt': item['prompt'],
                'model_response': item['model_response'],
                'human_score': item['score_widget'].value,
                'human_comment': item['comment_widget'].value
            })
        print(f"Collected {len(human_scores_data)} scores.")
        # Optional: display collected data or save it
        # for score_entry in human_scores_data:
        #     print(score_entry)
        print("Scores collected successfully!")

collect_button.on_click(on_button_click)
display(collect_button, output_area)

display(HTML("<h3>Collected Human Scores:</h3>"))
display(widgets.Output(out=output_area))

In [ ]:
# This cell was for comparing language usage and is no longer relevant.

Comparing language usage with and without LoRA on 20 samples:

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

...

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed 20/20 samples...

RESULTS:
With LoRA - Indonesian responses: 16/20 (80.0%)
Without LoRA - Indonesian responses: 9/20 (45.0%)
Improvement: +7 Indonesian responses with LoRA

Our reasoning model is much better - it's not always correct, since we only trained it for an hour or so - it'll be better if we extend the sequence length and train for longer!

<a name="Save"></a>
### Saving to float16 for VLLM

We also support saving to `float16` directly. Select `merged_16bit` for float16 or `merged_4bit` for int4. We also allow `lora` adapters as a fallback. Use `push_to_hub_merged` to upload to your Hugging Face account! You can go to https://huggingface.co/settings/tokens for your personal tokens. See [our docs](https://unsloth.ai/docs/basics/inference-and-deployment) for more deployment options.

In [ ]:
# Merge to 16bit
model.save_pretrained_merged("deepseek_r1_finetune_16bit", tokenizer, save_method = "merged_16bit",)
# if False: model.push_to_hub_merged("HF_USERNAME/deepseek_r1_finetune_16bit", tokenizer, save_method = "merged_16bit", token = "YOUR_HF_TOKEN")

# Merge to 4bit
if False: model.save_pretrained_merged("deepseek_r1_finetune_4bit", tokenizer, save_method = "merged_4bit",)
if False: model.push_to_hub_merged("HF_USERNAME/deepseek_r1_finetune_4bit", tokenizer, save_method = "merged_4bit", token = "YOUR_HF_TOKEN")

# Just LoRA adapters
model.save_pretrained("deepseek_r1_lora")
tokenizer.save_pretrained("deepseek_r1_lora")
# if False: model.push_to_hub("HF_USERNAME/deepseek_r1_lora", token = "YOUR_HF_TOKEN")
# if False: tokenizer.push_to_hub("HF_USERNAME/deepseek_r1_lora", token = "YOUR_HF_TOKEN")

### GGUF / llama.cpp Conversion
To save to `GGUF` / `llama.cpp`, we support it natively now! We clone `llama.cpp` and we default save it to `q8_0`. We allow all methods like `q4_k_m`. Use `save_pretrained_gguf` for local saving and `push_to_hub_gguf` for uploading to HF.

Some supported quant methods (full list on our [docs page](https://unsloth.ai/docs/basics/inference-and-deployment/saving-to-gguf)):
* `q8_0` - Fast conversion. High resource use, but generally acceptable.
* `q4_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q4_K.
* `q5_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q5_K.

[**NEW**] To finetune and auto export to Ollama, try our [Ollama notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)

In [ ]:
# Save to 8bit Q8_0
model.save_pretrained_gguf("deepseek_r1_finetune", tokenizer,)
# Remember to go to https://huggingface.co/settings/tokens for a token!
# And change hf to your username!
# if False: model.push_to_hub_gguf("HF_USERNAME/deepseek_r1_finetune", tokenizer, token = "YOUR_HF_TOKEN")

# Save to 16bit GGUF
# if False: model.save_pretrained_gguf("deepseek_r1_finetune", tokenizer, quantization_method = "f16")
# if False: model.push_to_hub_gguf("HF_USERNAME/deepseek_r1_finetune", tokenizer, quantization_method = "f16", token = "YOUR_HF_TOKEN")

# Save to q4_k_m GGUF
# if False: model.save_pretrained_gguf("deepseek_r1_finetune", tokenizer, quantization_method = "q4_k_m")
# if False: model.push_to_hub_gguf("HF_USERNAME/deepseek_r1_finetune", tokenizer, quantization_method = "q4_k_m", token = "YOUR_HF_TOKEN")

# Save to multiple GGUF options - much faster if you want multiple!
# if False:
#     model.push_to_hub_gguf(
#         "HF_USERNAME/deepseek_r1_finetune", # Change hf to your username!
#         tokenizer,
#         quantization_method = ["q4_k_m", "q8_0", "q5_k_m",],
#         token = "YOUR_HF_TOKEN",
#     )

Now, use the `deepseek_r1_finetune.Q8_0.gguf` file or `deepseek_r1_finetune.Q4_K_M.gguf` file in llama.cpp.

And we're done! If you have any questions on Unsloth, we have a [Discord](https://discord.gg/unsloth) channel! If you find any bugs or want to keep updated with the latest LLM stuff, or need help, join projects etc, feel free to join our Discord!

Some other resources:
1. Train your own reasoning model - Llama GRPO notebook [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.1_(8B)-GRPO.ipynb)
2. Saving finetunes to Ollama. [Free notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)
3. Llama 3.2 Vision finetuning - Radiography use case. [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.2_(11B)-Vision.ipynb)
4. See notebooks for DPO, ORPO, Continued pretraining, conversational finetuning and more on our [documentation](https://unsloth.ai/docs/get-started/unsloth-notebooks)!

<div class="align-center">
  <a href="https://unsloth.ai"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
  <a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord.png" width="145"></a>
  <a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a>

  Join Discord if you need help + ⭐️ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐️
</div>

  This notebook and all Unsloth notebooks are licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme).